<p align="center">
    <img src="https://beyondeducation.es/wp-content/uploads/2024/02/CUNEF-UNI50.png" width="500">
</p>


<h1 align="center"><b>Initial Assignment - Machine Learning [P375]</b></h1>
<h2 align="center">M2026-27 - Máster of Ciencia de Datos e Inteligencia Artificial (MCDIA-3)</h2>

| Full name | CUNEF email |
|---|---|
| Melanie Álvarez [second surname] | [melanie.alvarez]@cunef.edu |
| María Victoria del Campo [second surname] | [victoria.delcampo]@cunef.edu |
| Alicia Plaza [second surname] | [alicia.plaza]@cunef.edu |

**GitHub repository:** [https://github.com/mvdelcampo/machine-learning-project]

# **1. Business Problem Description**

Antimicrobial resistance (AMR) is a global public health crisis, it reduces therapeutic
options and complicates clinical decision-making, especially in acute care
settings. Ciprofloxacin (CIP) is a widely used fluoroquinolone antibiotic for treating various bacterial infections, particularly certain urinary tract infections. According to the European Centre for Disease Prevention and Control (ECDC), **30% of invasive *Escherichia coli* isolates in Spain were resistant to fluoroquinolones in 2024**, compared with the EU/EEA average of 22.5%, highlighting the relevance of investigating resistance to this antibiotic class.


### **Shortcomings of the current solution**

The conventional method for determining ciprofloxacin resistance is the antibiogram (culture and antibiotic susceptibility testing), which typically takes 48–72 hours. During this period, clinicians may need to prescribe empirical treatment without knowing whether the bacterial isolate is resistant to ciprofloxacin, potentially leading to ineffective therapy and delayed appropriate treatment. 


### **Opportunity**

Machine Learning can predict ciprofloxacin resistance using available susceptibility results from other antibiotics and patient characteristics, supporting clinical decision-making while the complete antibiogram is pending.


### **Limitations of previous work** 

Wishal and Sahara, *Predicting Multi-Drug Resistance in Bacterial Isolates Through Performance Comparison and LIME-based Interpretation of Classification Models*, used the same synthetic dataset to predict multidrug resistance (MDR), comparing five Machine Learning models (LR, RF, AdaBoost, XGBoost, LightGBM) and achieving performance metrics close to 99%. However, they used antibiotic resistance results both to define MDR target and as predictors, creating a risk of **target leakage** and making the problem almost deterministic and of little real predictive value.

### **Research question and technical formulation**
> ***Can the resistance status of ciprofloxacin be estimated from the susceptibility profiles of other antibiotics, bacterial taxa, and patient clinical features?***

- **Learning paradigm:** supervised binary classification
- **Target (y):** `CIP`, with **R (Resistent) = 1** and **S (Susceptible) = 0** for this initial operational definition.
- **Candidate features (X):** results from the remaining **14 antibiotics**, plus `Age`, `Gender`, `Souches`, `Diabetes`, `Hypertension`, `Hospital_before`, `Infection_Freq`.
- **Leakage guard:** neither `CIP` nor any feature calculated from CIP (such as MDR labels that include CIP) may enter X.

# **2. Data Source**

The dataset is the **Multi-resistance antibiotic susceptibility** dataset attributed to A. Hosni on Kaggle. The source file is downloaded from `https://www.kaggle.com/datasets/adilimadeddinehosni/multi-resistance-antibiotic-susceptibility/data`.

***Advantages:*** The dataset contains 10,710 bacterial isolates, including susceptibility results for 15 antibiotics and relevant clinical characteristics. Its size and variety of features make it suitable for investigating cross-resistance patterns and developing a binary classification model to predict ciprofloxacin resistance. Additionally, it is publicly available, supporting reproducibility.

***Disadvantages:*** The dataset is synthetic and simulates data from only one healthcare facility and includes a limited number of bacterial species, reducing its generalizability to real clinical environments. It also contains missing values, inconsistent categories, and limited clinical information.

# **3. Data loading**

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("./data")

bacteria_dataset = pd.read_csv(DATA_PATH / "Bacteria_dataset_Multiresictance.csv")
print("All datasets loaded successfully!")

All datasets loaded successfully!


# **4. Exploratory Data Analysis**

## Data description and inspection

In [134]:
bacteria_dataset.shape

(10710, 27)

In [135]:
bacteria_dataset.head()

,ID,Name,Email,Address,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,...,AN,Acide nalidixique,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Collection_Date,Notes
0,S290,Elizabeth Lawrence,elizabeth.lawrence@example.com,"6350 Robinson Loaf Apt. 447, Paulfurt, RI 30252",37/F,S290 Escherichia coli,No,No,No,0.0,...,S,S,S,S,R,S,S,S,5 Fev 2025,Patient had fever??
1,S291,Tina Sanders,tina.sanders@example.com,"78594 Galloway Port Suite 762, South Tanyatown...",29/F,S291 Morganella morganii,True,No,No,3.0,...,S,S,S,S,S,S,S,S,08/05/2021,تم العلاج
2,S292,Erin Cooke,erin.cooke@example.com,"76661 Isaiah Manors, North Benjamin, HI 85195",77/F,S292 Proteus mirabilis,True,No,No,3.0,...,S,S,S,R,R,S,S,S,01/05/2022,NADA
3,S293,Mrs. Sharon Watson,mrs..sharon.watson@example.com,"1670 Thompson Way Apt. 520, Andrewbury, IL 11534",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-05,PACIENTO ENFERMO
4,S294,Erika Stewart,erika.stewart@example.com,"76838 Patrick Green Suite 538, Torresmouth, ME...",13/F,S294 Escherichia coli,No,Yes,No,1.0,...,R,S,S,S,S,S,S,S,error,NADA


In [136]:
bacteria_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10710 entries, 0 to 10709
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ID                 10710 non-null  str  
 1   Name               10710 non-null  str  
 2   Email              10710 non-null  str  
 3   Address            10710 non-null  str  
 4   age/gender         10089 non-null  str  
 5   Souches            10070 non-null  str  
 6   Diabetes           10075 non-null  str  
 7   Hypertension       10080 non-null  str  
 8   Hospital_before    10073 non-null  str  
 9   Infection_Freq     10071 non-null  str  
 10  AMX/AMP            10052 non-null  str  
 11  AMC                10078 non-null  str  
 12  CZ                 10082 non-null  str  
 13  FOX                10085 non-null  str  
 14  CTX/CRO            10086 non-null  str  
 15  IPM                10077 non-null  str  
 16  GEN                10067 non-null  str  
 17  AN                 1008

The `Name`, `Email`, and `Address` columns contain personally identifiable information (PII) that could reveal a patient's identity. To protect patient privacy and follow data protection principles, this information should not be included in the analysis. Since these variables are not relevant to predicting antibiotic resistance, they are removed from the dataset.

In [137]:
# Drop columns containing personal information
bacteria_dataset.drop(columns=['Name', 'Email', 'Address'], inplace=True)

# Check results
bacteria_dataset.head()

/var/folders/w6/tt5g659n54zdxvlqlfd5dy3c0000gn/T/ipykernel_55902/3877739689.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  bacteria_dataset.drop(columns=['Name', 'Email', 'Address'], inplace=True)


,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,AN,Acide nalidixique,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Collection_Date,Notes
0,S290,37/F,S290 Escherichia coli,No,No,No,0.0,R,R,R,...,S,S,S,S,R,S,S,S,5 Fev 2025,Patient had fever??
1,S291,29/F,S291 Morganella morganii,True,No,No,3.0,S,R,S,...,S,S,S,S,S,S,S,S,08/05/2021,تم العلاج
2,S292,77/F,S292 Proteus mirabilis,True,No,No,3.0,S,R,S,...,S,S,S,R,R,S,S,S,01/05/2022,NADA
3,S293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-05,PACIENTO ENFERMO
4,S294,13/F,S294 Escherichia coli,No,Yes,No,1.0,S,S,R,...,R,S,S,S,S,S,S,S,error,NADA


### *Duplicates*

In [138]:
bacteria_dataset['ID'].nunique()

10710

### *Missing Values*

In [139]:
bacteria_dataset["age/gender"].unique()

<StringArray>
['37/F', '29/F', '77/F',    nan, '13/F', '57/M', '68/F', '18/F', '36/F',
 '49/M',
 ...
  '1/M', '67/M', '32/M', '23/M', '70/M', '37/M', '77/M', '10/M', '34/M',
 '66/M']
Length: 185, dtype: str

In [140]:
columna = bacteria_dataset["age/gender"].astype("string")

columna[
    ~columna.str.fullmatch(r"\d+\s*/\s*[FM]", case=False, na=False)
].unique()

<StringArray>
[<NA>, 'missing', '?']
Length: 3, dtype: string

In [141]:
age_gender = bacteria_dataset["age/gender"].astype("string").str.strip()
age_gender = age_gender.mask(
    age_gender.str.lower().isin(["missing", "nan", "?"])
)

bacteria_dataset[["Age", "Gender"]] = age_gender.str.split("/", expand=True)

In [142]:
bacteria_dataset['Age'] = pd.to_numeric(bacteria_dataset['Age'], errors='coerce')

In [143]:
bacteria_dataset.head()

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Collection_Date,Notes,Age,Gender
0,S290,37/F,S290 Escherichia coli,No,No,No,0.0,R,R,R,...,S,S,R,S,S,S,5 Fev 2025,Patient had fever??,37,F
1,S291,29/F,S291 Morganella morganii,True,No,No,3.0,S,R,S,...,S,S,S,S,S,S,08/05/2021,تم العلاج,29,F
2,S292,77/F,S292 Proteus mirabilis,True,No,No,3.0,S,R,S,...,S,R,R,S,S,S,01/05/2022,NADA,77,F
3,S293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-05,PACIENTO ENFERMO,<NA>,<NA>
4,S294,13/F,S294 Escherichia coli,No,Yes,No,1.0,S,S,R,...,S,S,S,S,S,S,error,NADA,13,F


In [144]:
bacteria_dataset["Collection_Date"].unique()

<StringArray>
['5 Fev 2025', '08/05/2021', '01/05/2022', '2020-01-05',      'error',
 '2020-12-31', '05/03/2022', '05/01/2025', '2023-12-31',          nan,
 '3 Jan 1019',          '?', '2024-12-31',    'missing']
Length: 14, dtype: str

In [145]:
bacteria_dataset["collection_year"] = (
    bacteria_dataset["Collection_Date"]
    .astype("string")
    .str.extract(r"(\d{4})", expand=False)
    .astype("Int64")
)

bacteria_dataset = bacteria_dataset.drop(columns="Collection_Date")

In [146]:
bacteria_dataset.head()

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Notes,Age,Gender,collection_year
0,S290,37/F,S290 Escherichia coli,No,No,No,0.0,R,R,R,...,S,S,R,S,S,S,Patient had fever??,37,F,2025
1,S291,29/F,S291 Morganella morganii,True,No,No,3.0,S,R,S,...,S,S,S,S,S,S,تم العلاج,29,F,2021
2,S292,77/F,S292 Proteus mirabilis,True,No,No,3.0,S,R,S,...,S,R,R,S,S,S,NADA,77,F,2022
3,S293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PACIENTO ENFERMO,<NA>,<NA>,2020
4,S294,13/F,S294 Escherichia coli,No,Yes,No,1.0,S,S,R,...,S,S,S,S,S,S,NADA,13,F,<NA>


In [147]:
mapping = {"true": 1, "yes": 1, "false": 0, "no": 0}

for col in ["Diabetes", "Hypertension", "Hospital_before"]:
    values = bacteria_dataset[col].astype("string").str.strip().str.lower()
    values = values.replace({"?": pd.NA, "missing": pd.NA, "": pd.NA})

    unexpected = set(values.dropna().unique()) - set(mapping)
    if unexpected:
        raise ValueError(f"Valores inesperados en {col}: {unexpected}")

    bacteria_dataset[col] = values.map(mapping).astype("Int64")

In [148]:
bacteria_dataset.head()

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Notes,Age,Gender,collection_year
0,S290,37/F,S290 Escherichia coli,0,0,0,0.0,R,R,R,...,S,S,R,S,S,S,Patient had fever??,37,F,2025
1,S291,29/F,S291 Morganella morganii,1,0,0,3.0,S,R,S,...,S,S,S,S,S,S,تم العلاج,29,F,2021
2,S292,77/F,S292 Proteus mirabilis,1,0,0,3.0,S,R,S,...,S,R,R,S,S,S,NADA,77,F,2022
3,S293,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PACIENTO ENFERMO,<NA>,<NA>,2020
4,S294,13/F,S294 Escherichia coli,0,1,0,1.0,S,S,R,...,S,S,S,S,S,S,NADA,13,F,<NA>


In [149]:
# Remove the first word, preserving single-word values
souches = bacteria_dataset["Souches"].astype("string").str.strip()

souches_sin_id = (
    souches
    .str.split(n=1)
    .str[1]
    .fillna(souches)
    .str.strip()
)

# Check unique values
print(souches_sin_id.unique())
print(souches_sin_id.value_counts(dropna=False))

['Escherichia coli' 'Morganella morganii' 'Proteus mirabilis' <NA>
 'Citrobacter spp.' 'Enterobacteria spp.' 'Proeus mirabilis'
 'Pseudomonas aeruginosa' 'Klebsiella pneumoniae' 'E. coli'
 'Klbsiella pneumoniae' 'E.cli' 'E.coi' 'Serratia marcescens'
 'Klebsie.lla pneumoniae' 'Acinetobacter baumannii' 'Prot.eus mirabilis'
 'missing' 'Protus mirabilis' 'Enter.bacteria spp.' '?'
 'Enteobacteria spp.']
Souches
Escherichia coli           4864
Enterobacteria spp.         802
<NA>                        640
Proteus mirabilis           598
Klebsiella pneumoniae       565
Citrobacter spp.            481
E.coi                       432
E.cli                       396
E. coli                     391
Morganella morganii         305
Serratia marcescens         256
Pseudomonas aeruginosa      200
Acinetobacter baumannii     181
Enter.bacteria spp.         100
Enteobacteria spp.           95
Klbsiella pneumoniae         74
Klebsie.lla pneumoniae       63
?                            63
missing       

In [150]:
# Dictionary to correct bacterial species names
corrections = {
    'E. coli': 'Escherichia coli',
    'E.cli': 'Escherichia coli',
    'E.coi': 'Escherichia coli',

    'Klbsiella pneumoniae': 'Klebsiella pneumoniae',
    'Klebsie.lla pneumoniae': 'Klebsiella pneumoniae',

    'Proeus mirabilis': 'Proteus mirabilis',
    'Prot.eus mirabilis': 'Proteus mirabilis',
    'Protus mirabilis': 'Proteus mirabilis',

    'Enter.bacteria spp.': 'Enterobacteria spp.',
    'Enteobacteria spp.': 'Enterobacteria spp.'
}

# Clean and standardize bacterial species
bacteria_dataset["Souches"] = (
    bacteria_dataset["Souches"]
    .astype("string")
    .str.strip()
    .str.replace(r"^S\d+\s+", "", regex=True)  # Remove sample ID
    .str.strip()
    .replace(corrections)  # Correct typos
    .replace(
        r"(?i)^(missing|na|nan|none|null|\?)$",
        pd.NA,
        regex=True
    )  # Replace missing indicators with pd.NA
)

In [151]:
print(bacteria_dataset["Souches"].unique())
print(bacteria_dataset["Souches"].nunique())
print(bacteria_dataset["Souches"].value_counts())

<StringArray>
[       'Escherichia coli',     'Morganella morganii',
       'Proteus mirabilis',                      <NA>,
        'Citrobacter spp.',     'Enterobacteria spp.',
  'Pseudomonas aeruginosa',   'Klebsiella pneumoniae',
     'Serratia marcescens', 'Acinetobacter baumannii']
Length: 10, dtype: string
9
Souches
Escherichia coli           6083
Enterobacteria spp.         997
Proteus mirabilis           742
Klebsiella pneumoniae       702
Citrobacter spp.            481
Morganella morganii         305
Serratia marcescens         256
Pseudomonas aeruginosa      200
Acinetobacter baumannii     181
Name: count, dtype: Int64


In [152]:
bacteria_dataset[bacteria_dataset["Souches"].isna()]

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Notes,Age,Gender,collection_year
3,S293,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PACIENTO ENFERMO,<NA>,<NA>,2020
14,S304,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,see file,<NA>,<NA>,2025
29,S319,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,--,<NA>,<NA>,2024
31,S321,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PACIENTO ENFERMO,<NA>,<NA>,2024
33,S323,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NADA,<NA>,<NA>,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10634,S10924,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,تم العلاج,<NA>,<NA>,2020
10646,S10936,NaN,<NA>,<NA>,<NA>,<NA>,?,NaN,NaN,NaN,...,missing,missing,missing,?,NaN,NaN,missing,<NA>,<NA>,<NA>
10650,S10940,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,--,<NA>,<NA>,2023
10687,S10977,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,MALADIE GRAVE,<NA>,<NA>,1019


In [153]:
bacteria_dataset["AMC"].value_counts(dropna=False)

AMC
R               5719
S               3876
NaN              632
i                 97
r                 96
s                 94
Intermediate      75
missing           66
?                 55
Name: count, dtype: int64

In [154]:
antibioticos = [
    "AMX/AMP", "AMC", "CZ", "FOX", "CTX/CRO", "IPM", "GEN", "AN",
    "Acide nalidixique", "ofx", "CIP", "C", "Co-trimoxazole",
    "Furanes", "colistine"
]

# Normalizar texto: quitar espacios y unificar mayúsculas
valores = bacteria_dataset[antibioticos].astype("string")
valores = valores.apply(lambda columna: columna.str.strip().str.upper())

# Estandarizar marcadores de ausencia y "Intermediate"
valores = valores.replace({
    "": pd.NA,
    "MISSING": pd.NA,
    "?": pd.NA,
    "NAN": pd.NA,
    "INTERMEDIATE": "I"
})

# Comprobar que no queden categorías distintas de R, S, I o ausente
inesperados = {
    columna: sorted(valores[columna].dropna().unique().tolist())
    for columna in antibioticos
    if not valores[columna].dropna().isin(["R", "S", "I"]).all()
}

if inesperados:
    raise ValueError(f"Valores inesperados en columnas de antibióticos: {inesperados}")

bacteria_dataset[antibioticos] = valores

In [155]:
bacteria_dataset

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Notes,Age,Gender,collection_year
0,S290,37/F,Escherichia coli,0,0,0,0.0,R,R,R,...,S,S,R,S,S,S,Patient had fever??,37,F,2025
1,S291,29/F,Morganella morganii,1,0,0,3.0,S,R,S,...,S,S,S,S,S,S,تم العلاج,29,F,2021
2,S292,77/F,Proteus mirabilis,1,0,0,3.0,S,R,S,...,S,R,R,S,S,S,NADA,77,F,2022
3,S293,NaN,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,PACIENTO ENFERMO,<NA>,<NA>,2020
4,S294,13/F,Escherichia coli,0,1,0,1.0,S,S,R,...,S,S,S,S,S,S,NADA,13,F,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10705,S10995,17/M,Enterobacteria spp.,0,0,0,3.0,S,R,R,...,S,S,S,S,S,S,see file,17,M,2024
10706,S10996,20/F,Proteus mirabilis,1,0,1,3.0,S,R,S,...,S,S,S,S,S,S,Patient had fever??,20,F,2025
10707,S10997,33/M,Escherichia coli,0,0,0,0.0,R,R,S,...,R,S,R,S,S,S,Patient had fever??,33,M,2021
10708,S10998,86/F,Escherichia coli,0,0,0,2.0,I,S,R,...,R,R,S,I,R,S,see file,86,F,2022


Outliers

Correlation

Normalization

Feature Engineering
- Disaggregation
- Categorization

Testing Hypothesis - Insights